# Code Converter, Optimizer & Summarizer

Switch between **OpenRouter (GPT-4o mini)** and **Ollama (Llama 3.2)** via the dropdown. **Gradio** provides the UI.

- **Convert**: Translate code between languages (e.g. Python → JavaScript).
- **Optimize**: Improve code for readability, performance, and best practices.
- **Summarize**: Generate a clear explanation of what the code does.

- **OpenRouter**: Set `OPENROUTER_API_KEY` in `.env` ([openrouter.ai](https://openrouter.ai)).
- **Ollama**: Run `ollama run llama3.2` locally; no API key needed.

Each response shows **how long** that model took (e.g. `⏱ openrouter-gpt-4o-mini: 2.34 s`), so you can compare **speed**. Judge **accuracy** by reviewing the converted, optimized, or summarized output.

In [ ]:
# Install dependencies if needed (uncomment in Colab or fresh env)
# %pip install -q openai python-dotenv gradio

In [6]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr

load_dotenv(override=True)

# OpenRouter: GPT-4o mini
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_client = None
if openrouter_api_key:
    openrouter_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
    print("OpenRouter connected (gpt-4o-mini)")

# Ollama: Llama 3.2 local
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
print("Ollama client configured (llama3.2 — ensure 'ollama run llama3.2' is running)")

# Model config: key -> (client, model_id)
MODELS = {}
if openrouter_client:
    MODELS["openrouter-gpt-4o-mini"] = (openrouter_client, "openai/gpt-4o-mini")
MODELS["ollama-llama3.2"] = (ollama_client, "llama3.2")

MODEL_CHOICES = list(MODELS.keys())
if not MODELS:
    raise RuntimeError("At least one backend required. Set OPENROUTER_API_KEY or run Ollama.")

OpenRouter connected (gpt-4o-mini)
Ollama client configured (llama3.2 — ensure 'ollama run llama3.2' is running)


In [7]:
def call_llm(model_key: str, system_prompt: str, user_content: str) -> str:
    """Call the selected LLM (OpenRouter or Ollama). Returns reply + timing or error string."""
    if model_key not in MODELS:
        return f"Unknown model: {model_key}. Choose from: {MODEL_CHOICES}"
    client, model_id = MODELS[model_key]
    try:
        start = time.perf_counter()
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            max_tokens=2048,
        )
        elapsed = time.perf_counter() - start
        content = (response.choices[0].message.content or "").strip()
        return content + f"\n\n--- ⏱ {model_key}: {elapsed:.2f} s ---"
    except Exception as e:
        return f"API error ({model_key}): {e}"

In [8]:
# System prompts for each task
CONVERT_SYSTEM = """You are an expert programmer. Convert the user's code to the target language they specify.
Output ONLY the converted code, with no extra explanation unless the user asks. Preserve logic and structure.
If the user does not specify a target language, convert to Python."""

OPTIMIZE_SYSTEM = """You are a senior software engineer. Optimize the user's code for:
- Readability and maintainability
- Performance where it matters
- Best practices and idiomatic style
Output the optimized code first, then a short bullet list of changes made."""

SUMMARIZE_SYSTEM = """You are a technical writer. Summarize the user's code in clear, concise language.
Explain what the code does, its main steps, inputs/outputs, and any notable behavior. Use plain English."""

In [9]:
def convert_code(model_key: str, code: str, target_lang: str) -> str:
    if not code.strip():
        return "Paste code and choose a target language."
    user_msg = f"Target language: {target_lang or 'Python'}\n\nCode:\n```\n{code}\n```"
    return call_llm(model_key, CONVERT_SYSTEM, user_msg)

def optimize_code(model_key: str, code: str) -> str:
    if not code.strip():
        return "Paste code to optimize."
    return call_llm(model_key, OPTIMIZE_SYSTEM, f"Code:\n```\n{code}\n```")

def summarize_code(model_key: str, code: str) -> str:
    if not code.strip():
        return "Paste code to summarize."
    return call_llm(model_key, SUMMARIZE_SYSTEM, f"Code:\n```\n{code}\n```")

## Gradio UI

In [10]:
with gr.Blocks(title="Code Converter, Optimizer & Summarizer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Code Converter, Optimizer & Summarizer")
    model_dropdown = gr.Dropdown(
        choices=MODEL_CHOICES,
        value=MODEL_CHOICES[0],
        label="LLM",
        info="OpenRouter (GPT-4o mini) or Ollama (Llama 3.2)",
    )

    with gr.Tabs():
        with gr.TabItem("Convert"):
            with gr.Row():
                code_convert = gr.Code(label="Source code", language="python", lines=14)
                out_convert = gr.Textbox(label="Converted code", lines=14)
            target_lang = gr.Dropdown(
                choices=["Python", "JavaScript", "TypeScript", "Java", "C#", "Go", "Rust", "Ruby"],
                value="JavaScript",
                label="Target language",
            )
            gr.Button("Convert").click(
                fn=convert_code,
                inputs=[model_dropdown, code_convert, target_lang],
                outputs=out_convert,
            )

        with gr.TabItem("Optimize"):
            with gr.Row():
                code_opt = gr.Code(label="Code to optimize", language="python", lines=14)
                out_opt = gr.Textbox(label="Optimized code & changes", lines=14)
            gr.Button("Optimize").click(
                fn=optimize_code,
                inputs=[model_dropdown, code_opt],
                outputs=out_opt,
            )

        with gr.TabItem("Summarize"):
            with gr.Row():
                code_sum = gr.Code(label="Code to summarize", language="python", lines=14)
                out_sum = gr.Textbox(label="Summary", lines=14)
            gr.Button("Summarize").click(
                fn=summarize_code,
                inputs=[model_dropdown, code_sum],
                outputs=out_sum,
            )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
